In [ ]:
"""Cell 1 - accelerator preflight (reused from dbald-gpu-probe). Confirm a real GPU BEFORE
spending the session on a ~31 GB download; enable_gpu is a request, not a guarantee (Kaggle hands out
CPU when the account's concurrent-GPU limit is taken). Abort early and cheaply if there is no GPU."""
import json, shutil, subprocess, time
r = {'t': time.time(), 'nvidia_smi_on_path': shutil.which('nvidia-smi') is not None}
try:
    r['nvidia_smi'] = subprocess.check_output(['nvidia-smi'], text=True)
except Exception as e:
    r['nvidia_smi'] = f'FAILED: {type(e).__name__}: {e}'
import torch
r['torch'] = torch.__version__
r['cuda_available'] = torch.cuda.is_available()
if torch.cuda.is_available():
    r['device_name'] = torch.cuda.get_device_name(0)
    r['total_vram_gib'] = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
print(json.dumps(r, indent=2))
if not torch.cuda.is_available():
    raise SystemExit('No GPU on this machine (enable_gpu ignored, most likely the concurrent-GPU '
                     'session limit is taken). Nothing was downloaded; re-run when a GPU is free.')


In [ ]:
"""Cell 2 - environment assembly (reused from dbald-research). Pinned deps, the UltraEdit
diffusers fork (which provides StableDiffusion3InstructPix2PixPipeline that our code imports), our
own repo at the exact probe commit, and the verified UltraEdit checkpoint. token=False throughout:
public access only, no credentials anywhere."""
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
TMP = Path('/kaggle/temp/probe'); TMP.mkdir(parents=True, exist_ok=True)
FORK_REV = '1af5f0478d56d62cf6f35781a2ab63e03d656b2d'
MODEL = 'BleachNick/SD3_UltraEdit_w_mask'
MODEL_REVISION = 'e5beecb69a3ef1399718be2f99b61730b8bf0346'
REPO_URL = 'https://github.com/hashimamjad-dev/DBALD-CVPR2026-Reproduction.git'
REPO_COMMIT = 'f236781e179e53b863d80663af38f22616c39b3b'
started = time.monotonic()

# 1) pinned deps, isolated (exact pins from dbald-research)
DEPS = TMP / 'dependencies'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(DEPS), '--no-deps',
                'transformers==4.56.2', 'accelerate==1.10.1', 'huggingface-hub==0.34.4',
                'tokenizers==0.22.2', 'sentencepiece==0.2.1', 'safetensors==0.6.2'],
               check=True, timeout=600)
sys.path.insert(0, str(DEPS))

# 2) UltraEdit diffusers fork -> the SD3 masked InstructPix2Pix pipeline
FORK = TMP / 'UltraEdit'
subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout',
                'https://github.com/HaozheZhao/UltraEdit.git', str(FORK)], check=True, timeout=300)
subprocess.run(['git', '-C', str(FORK), 'checkout', FORK_REV], check=True, timeout=300)
assert subprocess.check_output(['git', '-C', str(FORK), 'rev-parse', 'HEAD'], text=True).strip() == FORK_REV
sys.path.insert(0, str(FORK / 'diffusers/src'))
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

# 3) our repo at the exact probe commit (public clone, no creds)
REPO = TMP / 'dbald'
subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True, timeout=600)
subprocess.run(['git', '-C', str(REPO), 'checkout', REPO_COMMIT], check=True, timeout=120)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == REPO_COMMIT

# 4) verified UltraEdit checkpoint (public, token=False); import-sanity the pipeline resolves
from huggingface_hub import HfApi, snapshot_download
import diffusers
from diffusers import StableDiffusion3InstructPix2PixPipeline  # noqa: F401
info = HfApi(token=False).model_info(MODEL, revision=MODEL_REVISION, files_metadata=True)
required = sum(f.size or 0 for f in info.siblings)
free = shutil.disk_usage(TMP).free
if free < required + 10 * 1024**3:
    raise RuntimeError(f'Insufficient scratch: need {required/2**30:.1f}+10 GiB, have {free/2**30:.1f}')
SNAPSHOT = Path(snapshot_download(MODEL, revision=info.sha, cache_dir=TMP / 'cache',
                                  token=False, max_workers=2))
for entry in info.siblings:                                    # cheap byte-size verify (dbald-research)
    p = SNAPSHOT / entry.rfilename
    if p.stat().st_size != entry.size:
        raise ValueError('size mismatch: ' + entry.rfilename)
print(json.dumps(dict(diffusers_version=diffusers.__version__, diffusers_src=diffusers.__file__,
                      snapshot=str(SNAPSHOT), checkpoint_revision=info.sha, download_bytes=required,
                      repo_commit=REPO_COMMIT, assembly_seconds=round(time.monotonic() - started, 1)),
                 indent=2))


In [ ]:
"""Cell 3 - the timing sweep. Runs scripts/probe_diffusion_timing.py --run across the 6
SOURCES.md s9 configs on this T4, writing median s/image + peak VRAM per config vs the 6 s paper
target to /kaggle/working/timing.json. OOM-per-config is caught inside the probe, so a config too
big for a T4 (e.g. full-T5 resident) does not kill the sweep."""
import os, subprocess, sys
from pathlib import Path
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([str(FORK / 'diffusers/src'), str(DEPS), str(REPO / 'src')])
env['HF_HUB_OFFLINE'] = '1'                                    # load only from the local verified snapshot
env['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
out = Path('/kaggle/working/timing.json')
cmd = [sys.executable, str(REPO / 'scripts/probe_diffusion_timing.py'), '--run',
       '--snapshot', str(SNAPSHOT), '--warmup', '1', '--timed', '3', '--output', str(out)]
print('RUN:', ' '.join(cmd), flush=True)
rc = subprocess.run(cmd, env=env).returncode
print('probe return code:', rc, flush=True)
print(out.read_text() if out.exists() else 'NO timing.json produced')
